# RQ2 — Qwen S1/S3 trên toàn bộ 70 Lecture validation

**Hồ sơ lịch sử:** 70 bài này chỉ là tập con `Lecture` của 910 bản ghi `valid`. Theo quyết định 2026-10-07, EDA tiếp theo dùng toàn bộ `train`, `valid`, `test` không lọc genre; kết quả notebook này không đại diện cho phạm vi mới.

Chọn GPU trong Colab và chạy **Runtime → Run all** từ đầu. Notebook tự chứa mã, đọc đúng 70 DOI đã qua EDA ở split `valid`, rồi lưu từng cặp S1/S3 vào Drive. Chạy lại notebook sau khi Colab ngắt sẽ xác minh và dùng lại receipt hợp lệ. Không đọc `test` và không chạy LLM trên GPU máy cá nhân. Đây là đánh giá validation bằng abstract nhiễu; ROUGE không đo tính đúng nội dung. Ranh giới S3 do bộ phát hiện liên kết từ vựng chỉ dựa vào transcript tạo ra, chưa phải mô hình chia chương học được hay kết quả trên test kín.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.6', 'datasets==2.19.2',
                'huggingface_hub==0.36.2'], check=True)
print('Installed pinned notebook dependencies; PyTorch uses the Colab runtime installation.')

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/RQ2_TIB_VALID_QWEN')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
print('Results will be saved in:',DRIVE_ROOT)


## Mã chạy và công thức prompt đã ghim

Mã được ghi vào `/content/rq2_full70_runtime`. Hai nhánh dùng cùng mô hình và prompt; chỉ cách chia transcript khác nhau.


In [ ]:
from pathlib import Path
RUNTIME_ROOT=Path('/content/rq2_full70_runtime')
SCRIPT_DIR=RUNTIME_ROOT/'experiments/scripts'
SCRIPT_DIR.mkdir(parents=True,exist_ok=True)
(RUNTIME_ROOT/'experiments/__init__.py').write_text('',encoding='utf-8')
(SCRIPT_DIR/'__init__.py').write_text('',encoding='utf-8')
(SCRIPT_DIR/'rq2_controlled_validation.py').write_text('"""Fail-closed, validation-only RQ2 S1/S3 generation contract.\n\nThe repository\'s legacy engines expose neither exact usage nor retry receipts.\nThis runner accepts only a caller-supplied backend that reports every attempt.\nThe command-line entry point performs a safe cohort/backend preflight only.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\n\nROOT = Path(__file__).resolve().parents[2]\nRECEIPT = ROOT / "plans/261002-0946-rq2-controlled-summarization/rq2-tib-valid-diagnostic-receipt.json"\nDATASET = "gigant/tib"\nREVISION = "e6ab9fd7c6acd164f3284b8c4b17eb9d94681eee"\n\n\n@dataclass(frozen=True)\nclass RunConfig:\n    model_id: str\n    tokenizer_id: str\n    max_source_tokens: int\n    cumulative_prompt_budget: int\n    allowed_prompt_imbalance: int\n    chunk_tokens: int = 512\n    max_output_tokens: int = 128\n    final_output_tokens: int | None = None\n    prompt_recipe: str = "chat_faithful_full70_batched_v1"\n    map_batch_size: int = 1\n    temperature: float = 0.0\n    seed: int = 42\n    device: str = "unspecified"\n\n    def validate(self) -> None:\n        if not self.model_id or not self.tokenizer_id:\n            raise ValueError("A declared real model and exact tokenizer are required")\n        if min(self.max_source_tokens, self.cumulative_prompt_budget,\n               self.chunk_tokens, self.max_output_tokens) <= 0:\n            raise ValueError("Token budgets must be positive")\n        if self.final_output_tokens is not None and self.final_output_tokens <= 0:\n            raise ValueError("Final summary token budget must be positive")\n        if self.map_batch_size <= 0:\n            raise ValueError("Map batch size must be positive")\n        if self.prompt_recipe != "chat_faithful_full70_batched_v1":\n            raise ValueError("Unexpected pilot prompt recipe")\n        if self.allowed_prompt_imbalance < 0:\n            raise ValueError("Prompt imbalance tolerance must be nonnegative")\n        if not math.isfinite(self.temperature) or self.temperature < 0:\n            raise ValueError("Invalid decoding temperature")\n        if self.device not in {"unspecified", "cpu", "cuda"}:\n            raise ValueError("Unknown inference device")\n\n\nclass UsageBackend(Protocol):\n    model_id: str\n    tokenizer_id: str\n\n    def count_tokens(self, text: str) -> int: ...\n\n    def format_prompt(self, text: str) -> str: ...\n\n    def generate(self, prompt: str, *, max_output_tokens: int,\n                 temperature: float, seed: int) -> dict[str, Any]:\n        """Return text and attempts, each with exact prompt/output token counts."""\n\n\ndef load_declared_cohort(path: Path = RECEIPT) -> dict[str, Any]:\n    receipt = json.loads(path.read_text(encoding="utf-8"))\n    if (receipt.get("dataset") != DATASET or receipt.get("revision") != REVISION\n            or receipt.get("split_read") != "valid" or receipt.get("test_content_accessed") is not False):\n        raise ValueError("Only the pinned, validation-only TIB receipt is admissible")\n    selected = receipt.get("selected")\n    if not isinstance(selected, list) or len(selected) != 70:\n        raise ValueError("The declared 70-item cohort is missing")\n    dois = [item.get("doi") for item in selected]\n    if any(not isinstance(doi, str) or not doi for doi in dois) or len(set(dois)) != 70:\n        raise ValueError("Missing or duplicate selected DOI")\n    digest = hashlib.sha256(json.dumps(dois, ensure_ascii=False).encode()).hexdigest()\n    if digest != receipt.get("selected_doi_sha256"):\n        raise ValueError("Selected DOI order no longer matches its receipt digest")\n    return receipt\n\n\ndef preflight(config: RunConfig | None = None, backend: UsageBackend | None = None,\n              receipt_path: Path = RECEIPT) -> dict[str, Any]:\n    receipt = load_declared_cohort(receipt_path)\n    if config is None or backend is None:\n        return {"status": "blocked_backend_unavailable", "cohort_count": 70,\n                "selection_sha256": receipt["selected_doi_sha256"],\n                "reason": "No declared usage-reporting real model/backend was supplied"}\n    config.validate()\n    if backend.model_id != config.model_id or backend.tokenizer_id != config.tokenizer_id:\n        raise ValueError("Backend model/tokenizer differs from frozen configuration")\n    if not callable(getattr(backend, "count_tokens", None)) or not callable(getattr(backend, "generate", None)):\n        raise ValueError("Backend lacks exact tokenizer/generation methods")\n    if config.map_batch_size > 1 and (not callable(getattr(backend, "format_prompt", None))\n                                      or not callable(getattr(backend, "generate_batch", None))):\n        raise ValueError("Batched map generation requires format_prompt and generate_batch")\n    return {"status": "ready_for_validation_diagnostic", "cohort_count": 70,\n            "selection_sha256": receipt["selected_doi_sha256"], "model_id": config.model_id,\n            "tokenizer_id": config.tokenizer_id}\n\n\ndef _source(record: dict[str, Any], doi: str, backend: UsageBackend,\n            config: RunConfig) -> tuple[list[str], list[float], str]:\n    if record.get("doi") != doi:\n        raise ValueError("Record DOI is not the selected DOI")\n    segments = record.get("transcript_segments") or {}\n    texts, starts, ends = (segments.get(key) or [] for key in ("text", "start", "end"))\n    if not texts or len(texts) != len(starts) or len(texts) != len(ends):\n        raise ValueError("Missing or misaligned source transcript")\n    clean = [str(value).strip() for value in texts]\n    if any(not value for value in clean):\n        raise ValueError("Empty source segment")\n    if any(not isinstance(a, (int, float)) or not isinstance(b, (int, float))\n           or not math.isfinite(a) or not math.isfinite(b) or a < 0 or b < a\n           for a, b in zip(starts, ends)):\n        raise ValueError("Invalid source timestamps")\n    if any(starts[i] < starts[i - 1] for i in range(1, len(starts))):\n        raise ValueError("Overlapping or unordered source timestamps")\n    joined = " ".join(" ".join(clean).split())\n    count = backend.count_tokens(joined)\n    if not isinstance(count, int) or isinstance(count, bool) or count <= 0:\n        raise ValueError("Backend did not provide an exact source token count")\n    if count > config.max_source_tokens:\n        raise ValueError("Declared source span exceeds frozen source cap; no silent truncation")\n    return clean, [float(value) for value in starts], hashlib.sha256(joined.encode()).hexdigest()\n\n\ndef _fixed_chunks(text: str, backend: UsageBackend, limit: int) -> list[str]:\n    chunks: list[str] = []\n    current: list[str] = []\n    for word in text.split():\n        candidate = " ".join((*current, word))\n        if backend.count_tokens(candidate) > limit:\n            if not current:\n                raise ValueError("A source word exceeds the per-call token cap")\n            if backend.count_tokens(word) > limit:\n                raise ValueError("A source word exceeds the per-call token cap")\n            chunks.append(" ".join(current))\n            current = [word]\n        else:\n            current.append(word)\n    if current:\n        chunks.append(" ".join(current))\n    return chunks\n\n\ndef _hierarchy_chunks(texts: list[str], starts: list[float],\n                      boundaries: list[float], backend: UsageBackend, limit: int) -> list[str]:\n    if any(not isinstance(t, (int, float)) or not math.isfinite(t) for t in boundaries):\n        raise ValueError("Invalid predicted boundary")\n    if sorted(set(boundaries)) != boundaries:\n        raise ValueError("Predicted boundaries must be strictly ordered")\n    if any(t <= starts[0] or t >= starts[-1] for t in boundaries):\n        raise ValueError("Predicted boundary outside source span")\n    groups: list[list[str]] = [[] for _ in range(len(boundaries) + 1)]\n    for text, start in zip(texts, starts):\n        index = sum(start >= boundary for boundary in boundaries)\n        groups[index].append(text)\n    if any(not group for group in groups):\n        raise ValueError("Predicted boundary creates an empty chapter")\n    return [chunk for group in groups for chunk in _fixed_chunks(" ".join(group), backend, limit)]\n\n\ndef predict_source_boundaries(texts: list[str], starts: list[float]) -> list[float]:\n    """Small deterministic lexical-cohesion diagnostic; never reads chapter GT."""\n    if len(texts) < 3 or starts[-1] <= starts[0]:\n        return []\n    window = min(4, max(1, len(texts) // 8))\n    candidates = []\n    for index in range(1, len(texts)):\n        if not starts[0] + .08 * (starts[-1] - starts[0]) < starts[index] < starts[-1] - .08 * (starts[-1] - starts[0]):\n            continue\n        left = set(" ".join(texts[max(0, index - window):index]).casefold().split())\n        right = set(" ".join(texts[index:index + window]).casefold().split())\n        novelty = 1 - len(left & right) / max(1, len(left | right))\n        candidates.append((novelty, starts[index]))\n    target = min(5, max(1, int((starts[-1] - starts[0]) // 600)))\n    gap = max(1.0, (starts[-1] - starts[0]) / (target + 1) * .45)\n    chosen: list[float] = []\n    for _, timestamp in sorted(candidates, key=lambda item: (-item[0], item[1])):\n        if all(abs(timestamp - old) >= gap for old in chosen):\n            chosen.append(timestamp)\n        if len(chosen) >= target:\n            break\n    return sorted(chosen)\n\n\ndef _call(backend: UsageBackend, prompt: str, config: RunConfig,\n          calls: list[dict[str, Any]], *, output_limit: int | None = None) -> str:\n    limit = output_limit if output_limit is not None else config.max_output_tokens\n    formatted_prompt = backend.format_prompt(prompt)\n    response = backend.generate(formatted_prompt, max_output_tokens=limit,\n                                temperature=config.temperature, seed=config.seed)\n    if not isinstance(response, dict) or not isinstance(response.get("text"), str) or not response["text"].strip():\n        raise ValueError("Backend returned empty or malformed output")\n    attempts = response.get("attempts")\n    if not isinstance(attempts, list) or not attempts:\n        raise ValueError("Backend omitted exact per-attempt usage")\n    expected_prompt = backend.count_tokens(formatted_prompt)\n    for attempt in attempts:\n        if not isinstance(attempt, dict):\n            raise ValueError("Malformed backend attempt")\n        p, o = attempt.get("prompt_tokens"), attempt.get("output_tokens")\n        if (not isinstance(p, int) or isinstance(p, bool) or p != expected_prompt\n                or not isinstance(o, int) or isinstance(o, bool) or o < 0\n                or o > limit):\n            raise ValueError("Backend usage disagrees with exact tokenizer or is incomplete")\n        if attempt.get("status") not in {"retry", "failed", "success"}:\n            raise ValueError("Backend omitted an attempt status")\n        calls.append({"prompt_tokens": p, "output_tokens": o,\n                      "content_tokens": attempt.get("content_tokens"),\n                      "status": attempt.get("status", "unknown")})\n    if attempts[-1].get("status") != "success":\n        raise ValueError("Last generation attempt did not succeed")\n    # Decode and re-encode need not preserve token count. Check each count\n    # against its own source: raw generation usage and visible text.\n    visible_count = backend.count_tokens(response["text"])\n    final = attempts[-1]\n    if final.get("content_tokens", final["output_tokens"]) != visible_count:\n        raise ValueError("Backend final text count disagrees with exact tokenizer")\n    if sum(call["prompt_tokens"] for call in calls) > config.cumulative_prompt_budget:\n        raise ValueError("Cumulative prompt budget exceeded, including retries")\n    return response["text"].strip()\n\n\ndef _arm(chunks: list[str], backend: UsageBackend, config: RunConfig) -> dict[str, Any]:\n    calls: list[dict[str, Any]] = []\n    prompts = []\n    for index, chunk in enumerate(chunks, 1):\n        prompt = (f"Lecture section {index}:\\n{chunk}\\n\\n"\n                  "Write 2 or 3 complete English sentences about the main subject and claims "\n                  "in this section. Use only this source. Prefer subject matter over greetings "\n                  "or meeting logistics. Preserve technical terms, numbers, and chronology exactly; do not expand unclear acronyms. End with a complete sentence.")\n        prompts.append(prompt)\n    sections = []\n    if config.map_batch_size == 1:\n        sections = [_call(backend, prompt, config, calls) for prompt in prompts]\n    else:\n        for start in range(0, len(prompts), config.map_batch_size):\n            formatted = [backend.format_prompt(prompt) for prompt in prompts[start:start + config.map_batch_size]]\n            responses = backend.generate_batch(formatted, max_output_tokens=config.max_output_tokens,\n                                               temperature=config.temperature, seed=config.seed)\n            if not isinstance(responses, list) or len(responses) != len(formatted):\n                raise ValueError("Batch backend returned the wrong number of responses")\n            for formatted_prompt, response in zip(formatted, responses):\n                if not isinstance(response, dict) or not isinstance(response.get("text"), str) or not response["text"].strip():\n                    raise ValueError("Batch backend returned empty or malformed output")\n                attempts = response.get("attempts")\n                if (not isinstance(attempts, list) or not attempts\n                        or not isinstance(attempts[-1], dict)\n                        or attempts[-1].get("status") != "success"):\n                    raise ValueError("Batch backend omitted successful per-attempt usage")\n                expected_prompt = backend.count_tokens(formatted_prompt)\n                for attempt in attempts:\n                    if not isinstance(attempt, dict):\n                        raise ValueError("Malformed batch backend attempt")\n                    p, o = attempt.get("prompt_tokens"), attempt.get("output_tokens")\n                    if (not isinstance(p, int) or isinstance(p, bool) or p != expected_prompt\n                            or not isinstance(o, int) or isinstance(o, bool)\n                            or o < 0 or o > config.max_output_tokens\n                            or attempt.get("status") not in {"retry", "failed", "success"}):\n                        raise ValueError("Batch backend usage disagrees with exact tokenizer")\n                    calls.append({"prompt_tokens": p, "output_tokens": o,\n                                  "content_tokens": attempt.get("content_tokens"),\n                                  "status": attempt["status"]})\n                if attempts[-1].get("content_tokens", attempts[-1]["output_tokens"]) != backend.count_tokens(response["text"]):\n                    raise ValueError("Batch backend text count disagrees with tokenizer")\n                if sum(call["prompt_tokens"] for call in calls) > config.cumulative_prompt_budget:\n                    raise ValueError("Cumulative prompt budget exceeded, including retries")\n                sections.append(response["text"].strip())\n    reduce_prompt = ("These are notes from consecutive sections of one lecture.\\n"\n                     + "\\n".join(f"Section {i}: {part}" for i, part in enumerate(sections, 1))\n                     + "\\n\\nWrite one coherent 120-180 word English lecture summary. "\n                       "Cover the main subject and key claims across the lecture, not just the last section. "\n                       "Use only these notes. Omit greetings, logistics, section numbers, "\n                       "and comments about the summary. Preserve technical terms, numbers, and chronology; omit unclear or conflicting details. End with a complete sentence.")\n    summary = _call(backend, reduce_prompt, config, calls,\n                    output_limit=config.final_output_tokens)\n    return {"summary": summary, "calls": calls,\n            "prompt_tokens": sum(call["prompt_tokens"] for call in calls),\n            "output_tokens": sum(call["output_tokens"] for call in calls),\n            "chunk_count": len(chunks),\n            "map_gpu_batches": math.ceil(len(chunks) / config.map_batch_size)}\n\n\ndef _receipt_hash(result: dict[str, Any]) -> str:\n    data = {key: value for key, value in result.items() if key != "receipt_sha256"}\n    return hashlib.sha256(json.dumps(data, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\n\n\ndef run_pair(record: dict[str, Any], doi: str, boundaries: list[float],\n             backend: UsageBackend, config: RunConfig, output_dir: Path,\n             receipt_path: Path = RECEIPT) -> dict[str, Any]:\n    gate = preflight(config, backend, receipt_path)\n    receipt = load_declared_cohort(receipt_path)\n    if doi not in {item["doi"] for item in receipt["selected"]}:\n        raise ValueError("DOI is outside declared validation cohort")\n    texts, starts, source_hash = _source(record, doi, backend, config)\n    predicted = predict_source_boundaries(texts, starts)\n    if not predicted or boundaries != predicted:\n        raise ValueError("S3 requires the runner\'s source-only predicted boundaries")\n    configuration = vars(config)\n    identity = hashlib.sha256(json.dumps({"doi": doi, "source": source_hash,\n        "boundaries": boundaries, "config": configuration,\n        "selection": gate["selection_sha256"]}, sort_keys=True).encode()).hexdigest()\n    path = output_dir / f"{re.sub(r\'[^a-zA-Z0-9]+\', \'-\', doi).strip(\'-\')}.json"\n    if path.exists():\n        old = json.loads(path.read_text(encoding="utf-8"))\n        if (not isinstance(old, dict) or old.get("identity_sha256") != identity\n                or old.get("status") != "paired_diagnostic"\n                or old.get("receipt_sha256") != _receipt_hash(old)\n                or set(old.get("arms", {})) != {"S1", "S3"}):\n            raise ValueError("Existing DOI receipt differs; refusing unsafe resume")\n        return old\n    source_text = " ".join(" ".join(texts).split())\n    s1_chunks = _fixed_chunks(source_text, backend, config.chunk_tokens)\n    s3_chunks = _hierarchy_chunks(texts, starts, boundaries, backend, config.chunk_tokens)\n    if " ".join(s1_chunks) != source_text or " ".join(s3_chunks) != source_text:\n        raise ValueError("S1/S3 do not cover the identical source span")\n    s1 = _arm(s1_chunks, backend, config)\n    s3 = _arm(s3_chunks, backend, config)\n    imbalance = abs(s1["prompt_tokens"] - s3["prompt_tokens"])\n    if imbalance > config.allowed_prompt_imbalance:\n        raise ValueError("Cumulative prompt-token imbalance exceeds frozen tolerance")\n    result = {"status": "paired_diagnostic", "dataset": DATASET, "revision": REVISION,\n              "split": "valid", "test_content_accessed": False, "doi": doi,\n              "identity_sha256": identity, "selection_sha256": gate["selection_sha256"],\n              "transcript_sha256": source_hash, "source_tokens": backend.count_tokens(source_text),\n              "predicted_boundaries_seconds": boundaries,\n              "boundary_predictor": "source_only_adjacent_window_lexical_cohesion_v1",\n              "configuration": configuration,\n              "prompt_imbalance_tokens": imbalance, "arms": {"S1": s1, "S3": s3},\n              "interpretation": "Engineering diagnostic; no human factual-support judgment or formal H2 claim."}\n    result["receipt_sha256"] = _receipt_hash(result)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(".tmp")\n    temporary.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    os.replace(temporary, path)\n    return result\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--receipt", type=Path, default=RECEIPT)\n    args = parser.parse_args()\n    print(json.dumps(preflight(receipt_path=args.receipt), indent=2))\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')
(SCRIPT_DIR/'rq2_local_hf_backend.py').write_text('"""Local, pinned Transformers backend with exact token receipts for RQ2.\n\nThis adapter uses raw prompt text consistently for counting and generation.\nIt never fetches model files: the supplied directory must already be complete.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\n\nclass LocalHfUsageBackend:\n    def __init__(self, model_dir: Path, *, model_id: str, revision: str,\n                 device: str = "cpu") -> None:\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n\n        folder = Path(model_dir).resolve()\n        if not folder.is_dir() or not (folder / "model.safetensors").is_file():\n            raise FileNotFoundError(f"Pinned local model weights are missing: {folder}")\n        if not model_id or not revision:\n            raise ValueError("Model identity and pinned revision are required")\n        if device not in {"cpu", "cuda"}:\n            raise ValueError("Device must be cpu or cuda")\n        if device == "cuda" and not torch.cuda.is_available():\n            raise RuntimeError("CUDA is unavailable; do not retry a lost GPU device")\n        self.model_id = f"{model_id}@{revision}"\n        self.tokenizer_id = self.model_id\n        self.tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True)\n        self.tokenizer.padding_side = "left"\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.model = AutoModelForCausalLM.from_pretrained(\n            folder, local_files_only=True, torch_dtype=torch.float16\n        )\n        self.device = device\n        self.model.to(self.device)\n        self.model.eval()\n        self.context_limit = int(self.model.config.max_position_embeddings)\n\n    def count_tokens(self, text: str) -> int:\n        return len(self.tokenizer.encode(text, add_special_tokens=False))\n\n    def format_prompt(self, text: str) -> str:\n        return self.tokenizer.apply_chat_template(\n            [{"role": "user", "content": text}],\n            tokenize=False, add_generation_prompt=True\n        )\n\n    def generate(self, prompt: str, *, max_output_tokens: int,\n                 temperature: float, seed: int) -> dict:\n        import torch\n\n        if temperature != 0:\n            raise ValueError("This controlled backend currently supports greedy decoding only")\n        inputs = self.tokenizer(prompt, return_tensors="pt", add_special_tokens=False)\n        input_ids = inputs["input_ids"].to(self.device)\n        prompt_tokens = int(input_ids.shape[-1])\n        if prompt_tokens + max_output_tokens > self.context_limit:\n            raise ValueError("Prompt plus output budget exceeds pinned model context")\n        torch.manual_seed(seed)\n        if self.device == "cuda":\n            torch.cuda.manual_seed_all(seed)\n        with torch.inference_mode():\n            output = self.model.generate(\n                input_ids=input_ids,\n                attention_mask=inputs["attention_mask"].to(self.device),\n                max_new_tokens=max_output_tokens,\n                do_sample=False,\n                pad_token_id=self.tokenizer.eos_token_id,\n            )\n        generated = output[0, prompt_tokens:]\n        text = self.tokenizer.decode(generated, skip_special_tokens=True).strip()\n        return {\n            "text": text,\n            "attempts": [{"status": "success", "prompt_tokens": prompt_tokens,\n                          "output_tokens": int(generated.shape[-1]),\n                          "content_tokens": self.count_tokens(text)}],\n        }\n\n    def generate_batch(self, prompts: list[str], *, max_output_tokens: int,\n                       temperature: float, seed: int) -> list[dict]:\n        """Generate several map summaries in one GPU call with exact per-prompt usage."""\n        import torch\n\n        if not prompts:\n            return []\n        if temperature != 0:\n            raise ValueError("This controlled backend supports greedy decoding only")\n        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True,\n                                add_special_tokens=False)\n        input_ids = inputs["input_ids"].to(self.device)\n        attention_mask = inputs["attention_mask"].to(self.device)\n        padded_length = int(input_ids.shape[-1])\n        if padded_length + max_output_tokens > self.context_limit:\n            raise ValueError("Batched prompt plus output budget exceeds pinned model context")\n        prompt_counts = [int(value) for value in attention_mask.sum(dim=1).tolist()]\n        torch.manual_seed(seed)\n        if self.device == "cuda":\n            torch.cuda.manual_seed_all(seed)\n        with torch.inference_mode():\n            output = self.model.generate(\n                input_ids=input_ids, attention_mask=attention_mask,\n                max_new_tokens=max_output_tokens, do_sample=False,\n                pad_token_id=self.tokenizer.pad_token_id,\n            )\n        responses = []\n        for row, prompt_tokens in zip(output[:, padded_length:], prompt_counts):\n            generated = row.tolist()\n            if self.tokenizer.eos_token_id in generated:\n                generated = generated[:generated.index(self.tokenizer.eos_token_id) + 1]\n            text = self.tokenizer.decode(generated, skip_special_tokens=True).strip()\n            responses.append({\n                "text": text,\n                "attempts": [{"status": "success", "prompt_tokens": prompt_tokens,\n                              "output_tokens": len(generated),\n                              "content_tokens": self.count_tokens(text)}],\n            })\n        return responses\n',encoding='utf-8')
print('Pinned runner and backend written to',SCRIPT_DIR)


In [ ]:
import hashlib,json,sys
from pathlib import Path
SELECTED_DOIS=['10.5446/15130 (DOI)', '10.5446/20846 (DOI)', '10.5446/18990 (DOI)', '10.5446/49062 (DOI)', '10.5446/49469 (DOI)', '10.5446/21651 (DOI)', '10.5446/21609 (DOI)', '10.14288/1.0398145 (DOI)', '10.5446/43747 (DOI)', '10.5446/19001 (DOI)', '10.5446/49487 (DOI)', '10.5446/15101 (DOI)', '10.5446/38638 (DOI)', '10.5446/49452 (DOI)', '10.5446/35254 (DOI)', '10.5446/19366 (DOI)', '10.5446/54077 (DOI)', '10.5446/42682 (DOI)', '10.5446/49509 (DOI)', '10.5446/50943 (DOI)', '10.5446/19013 (DOI)', '10.5446/55051 (DOI)', '10.5446/50930 (DOI)', '10.5446/16990 (DOI)', '10.5446/49070 (DOI)', '10.5446/38319 (DOI)', '10.5446/14078 (DOI)', '10.5446/42583 (DOI)', '10.5446/15120 (DOI)', '10.5446/19407 (DOI)', '10.5446/19464 (DOI)', '10.5446/15074 (DOI)', '10.5446/19015 (DOI)', '10.5446/43742 (DOI)', '10.5446/15106 (DOI)', '10.5446/43757 (DOI)', '10.5446/16294 (DOI)', '10.5446/50934 (DOI)', '10.5446/49089 (DOI)', '10.5446/19460 (DOI)', '10.5446/20231 (DOI)', '10.5446/48906 (DOI)', '10.5446/46066 (DOI)', '10.5446/46055 (DOI)', '10.5446/42685 (DOI)', '10.5446/18183 (DOI)', '10.5446/42568 (DOI)', '10.5446/19355 (DOI)', '10.5446/21634 (DOI)', '10.5446/18890 (DOI)', '10.5446/19217 (DOI)', '10.5446/20848 (DOI)', '10.5446/18344 (DOI)', '10.5446/48908 (DOI)', '10.5446/15021 (DOI)', '10.5446/49060 (DOI)', '10.5446/55669 (DOI)', '10.5446/18588 (DOI)', '10.5446/42637 (DOI)', '10.5446/37055 (DOI)', '10.5446/14934 (DOI)', '10.5446/19397 (DOI)', '10.5446/45978 (DOI)', '10.5446/49247 (DOI)', '10.5446/15109 (DOI)', '10.5446/16394 (DOI)', '10.5446/19395 (DOI)', '10.5446/17056 (DOI)', '10.5446/16219 (DOI)', '10.5446/55054 (DOI)']
SELECTION_SHA256='e4c17abba5ac7f9b406c674b96d5c3f1aa64c1dbf5cd1601d2f03181bdcb0cbd'
assert hashlib.sha256(json.dumps(SELECTED_DOIS,ensure_ascii=False).encode()).hexdigest()==SELECTION_SHA256
assert len(SELECTED_DOIS)==len(set(SELECTED_DOIS))==70
receipt={'dataset':'gigant/tib','revision':'e6ab9fd7c6acd164f3284b8c4b17eb9d94681eee','split_read':'valid','test_content_accessed':False,'selected':[{'doi':doi} for doi in SELECTED_DOIS],'selected_doi_sha256':SELECTION_SHA256}
receipt_path=RUNTIME_ROOT/'plans/261002-0946-rq2-controlled-summarization/rq2-tib-valid-diagnostic-receipt.json'
receipt_path.parent.mkdir(parents=True,exist_ok=True)
receipt_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
sys.path.insert(0,str(RUNTIME_ROOT))
print('Pinned valid cohort:',len(SELECTED_DOIS),'DOI; hash:',SELECTION_SHA256)


## Đọc 70 bài từ split `valid`

Đọc văn bản và metadata theo revision đã ghim; chưa tải trọng số mô hình.


In [ ]:
from datasets import load_dataset
DATA_REVISION='e6ab9fd7c6acd164f3284b8c4b17eb9d94681eee'
wanted=set(SELECTED_DOIS)
records={}
for record in load_dataset('gigant/tib',split='valid',revision=DATA_REVISION,streaming=True):
    doi=str(record.get('doi') or '').strip()
    if doi in wanted:
        records[doi]=record
        if len(records)==len(wanted):break
assert set(records)==wanted, 'Pinned valid cohort is incomplete; no generation started.'
print('Loaded',len(records),'validation records. Sealed test was not accessed.')


## Đối chiếu nguồn và abstract

Xuất transcript/abstract của đủ 70 bài để đối chiếu sau chạy. Bảng hiển thị độ dài và số đoạn từng bài; không gán nhãn chất lượng.


In [ ]:
import hashlib,re,pandas as pd
from IPython.display import display

source_rows=[]
for doi in SELECTED_DOIS:
    record=records[doi]
    segments=record['transcript_segments']
    texts=[str(value).strip() for value in segments['text']]
    starts=[float(value) for value in segments['start']]
    ends=[float(value) for value in segments['end']]
    assert len(texts)==len(starts)==len(ends) and len(texts)>=2
    assert all(starts[i]>=starts[i-1] for i in range(1,len(starts)))
    source_text=' '.join(' '.join(texts).split())
    abstract=str(record.get('abstract') or '').strip()
    source_rows.append({'doi':doi,'title':str(record.get('title') or ''),
        'subject':str(record.get('subject') or ''),'author':str(record.get('author') or ''),
        'abstract':abstract,'abstract_words':len(abstract.split()),
        'source_words':len(source_text.split()),
        'source_sha256':hashlib.sha256(source_text.encode()).hexdigest(),
        'segment_count':len(texts),'duration_seconds':round(ends[-1]-starts[0],2),
        'transcript_segments':{'text':texts,'start':starts,'end':ends}})
source_review_path=DRIVE_ROOT/'rq2_tib_valid_70_source_reference_review.json'
source_review={'status':'source_reference_review_only','dataset':'gigant/tib',
    'revision':DATA_REVISION,'split':'valid','test_content_accessed':False,
    'selection_sha256':SELECTION_SHA256,'records':source_rows}
source_review_path.write_text(json.dumps(source_review,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
display(pd.DataFrame([{key:row[key] for key in ('doi','subject','abstract_words',
    'source_words','segment_count','duration_seconds')} for row in source_rows]))
print('Source/reference snapshot saved:',source_review_path)


## Tải Qwen trên GPU Colab

Nếu vừa đổi runtime từ CPU sang GPU, chạy lại notebook từ đầu.


In [ ]:
import torch,importlib,sys
from huggingface_hub import snapshot_download
assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → GPU, then reconnect.'
print('Colab GPU:',torch.cuda.get_device_name(0))
for module_name in ('experiments.scripts.rq2_controlled_validation',
                    'experiments.scripts.rq2_local_hf_backend',
                    'experiments.scripts','experiments'):
    sys.modules.pop(module_name,None)
importlib.invalidate_caches()
from experiments.scripts.rq2_controlled_validation import (
    RunConfig,load_declared_cohort,predict_source_boundaries,preflight,run_pair)
from experiments.scripts.rq2_local_hf_backend import LocalHfUsageBackend
assert 'prompt_recipe' in RunConfig.__dataclass_fields__, 'Pilot runner did not load; rerun the code-writing cell'
MODEL_ID='Qwen/Qwen2.5-1.5B-Instruct'
MODEL_REVISION='989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
model_dir=Path('/content/rq2_models/qwen2_5_1_5b_instruct')
model_dir.mkdir(parents=True,exist_ok=True)
snapshot_download(MODEL_ID,revision=MODEL_REVISION,local_dir=model_dir,
                  allow_patterns=['config.json','generation_config.json','tokenizer*',
                                  'special_tokens_map.json','model.safetensors'],max_workers=2)
backend=LocalHfUsageBackend(model_dir,model_id=MODEL_ID,revision=MODEL_REVISION,device='cuda')
cohort=load_declared_cohort()
print('Pinned full-70 model loaded; validation source cohort:',len(records))


## Chạy toàn bộ 70 cặp và lưu từng bài

Cùng Qwen, greedy decoding và chat template. Map 160 token, bản cuối 640 token; gom tối đa 4 đoạn vào mỗi lần sinh trên T4 bằng padding bên trái. prompt yêu cầu giữ đúng thuật ngữ, số và trình tự thời gian. Notebook kiểm tra số token nguồn trước khi sinh, ghi manifest sau mỗi bài và tự tiếp tục receipt hợp lệ. Các DOI lỗi vẫn nằm trong bảng kết quả.


In [ ]:
import hashlib,json,re,time,os,shutil
RUN_LIMIT=len(SELECTED_DOIS)
config=RunConfig(model_id=backend.model_id,tokenizer_id=backend.tokenizer_id,
    max_source_tokens=65536,cumulative_prompt_budget=131072,
    allowed_prompt_imbalance=2048,chunk_tokens=512,max_output_tokens=160,
    final_output_tokens=640,map_batch_size=4,temperature=0.0,seed=42,device='cuda')
gate=preflight(config,backend)
assert gate['cohort_count']==RUN_LIMIT==70
source_tokens={doi:backend.count_tokens(' '.join(
    ' '.join(str(t).strip() for t in records[doi]['transcript_segments']['text']).split()))
    for doi in SELECTED_DOIS}
display(pd.DataFrame([{'doi':doi,'source_tokens':source_tokens[doi],
    'within_source_cap':source_tokens[doi]<=config.max_source_tokens}
    for doi in SELECTED_DOIS]))
assert all(n<=config.max_source_tokens for n in source_tokens.values()), 'Source cap fails; generation not started.'
run_key=hashlib.sha256(json.dumps({'config':vars(config),'cohort':SELECTION_SHA256},
    sort_keys=True).encode()).hexdigest()[:12]
output_dir=DRIVE_ROOT/f'full70_chat_{run_key}'
output_dir.mkdir(parents=True,exist_ok=True)
shutil.copy2(source_review_path,output_dir/'source_reference_70_valid.json')
manifest_path=output_dir/'manifest.json'
order=sorted(SELECTED_DOIS,key=lambda doi:(source_tokens[doi],doi))
existing={}
if manifest_path.exists():
    old=json.loads(manifest_path.read_text(encoding='utf-8'))
    assert (old['selection_sha256']==SELECTION_SHA256 and old['config']==vars(config)
            and old['declared_dois']==SELECTED_DOIS), 'Saved run uses another cohort/config.'
    existing={item['doi']:item for item in old['results']}

def save_manifest():
    all_rows=[existing.get(doi,{'doi':doi,'status':'not_run','reason':'Not attempted yet'})
              for doi in SELECTED_DOIS]
    manifest={'status':'validation_full70_diagnostic','dataset':'gigant/tib',
      'revision':DATA_REVISION,'split':'valid','test_content_accessed':False,
      'model':backend.model_id,'selection_sha256':SELECTION_SHA256,
      'config':vars(config),'requested_limit':RUN_LIMIT,'declared_dois':SELECTED_DOIS,
      'results':all_rows}
    tmp=manifest_path.with_suffix('.tmp')
    tmp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    os.replace(tmp,manifest_path)
    return manifest

save_manifest()
for number,doi in enumerate(order,1):
    record=records[doi]; segments=record['transcript_segments']; started=time.perf_counter()
    try:
        boundaries=predict_source_boundaries([str(v).strip() for v in segments['text']],
                                            [float(v) for v in segments['start']])
        pair=run_pair(record,doi,boundaries,backend,config,output_dir)
        if existing.get(doi,{}).get('status')=='completed':
            print(f'{number}/{RUN_LIMIT}: {doi} -> resumed (previous GPU time retained)',flush=True)
            continue
        name=re.sub(r'[^a-zA-Z0-9]+','-',doi).strip('-')+'.json'
        item={'doi':doi,'status':'completed','path':str(output_dir/name),
              'source_tokens':pair['source_tokens'],
              'prompt_imbalance_tokens':pair['prompt_imbalance_tokens']}
    except (ValueError,RuntimeError,OSError) as exc:
        item={'doi':doi,'status':'failed','error_type':type(exc).__name__,'reason':str(exc)}
    item['seconds']=round(time.perf_counter()-started,2)
    existing[doi]=item; save_manifest()
    print(f'{number}/{RUN_LIMIT}: {doi} -> {item["status"]}; {item["seconds"]} s',flush=True)
print('Manifest:',manifest_path)


## Đánh giá tự động

Bảng có đủ 70 DOI và trạng thái từng bài. ROUGE-L dùng abstract do nguồn cung cấp làm tham chiếu nhiễu; chênh lệch S3−S1 và bootstrap chỉ tính trên cặp hoàn thành có abstract. Đọc số lần chạm trần token và lý do lỗi cùng điểm số.


In [ ]:
import random,statistics,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
def words(text):
    return re.findall(r"[a-z0-9']+",(text or '').lower())
def rouge_l_f1(reference,hypothesis):
    a,b=words(reference),words(hypothesis)
    if not a or not b:return 0.0
    prev=[0]*(len(b)+1)
    for token in a:
        curr=[0]*(len(b)+1)
        for j,other in enumerate(b,1):
            curr[j]=prev[j-1]+1 if token==other else max(prev[j],curr[j-1])
        prev=curr
    lcs=prev[-1]; precision=lcs/len(b); recall=lcs/len(a)
    return 2*precision*recall/(precision+recall) if precision+recall else 0.0
def paired_ci(differences,seed=20261002,samples=5000):
    if not differences:return None
    rng=random.Random(seed); n=len(differences)
    means=sorted(sum(differences[rng.randrange(n)] for _ in range(n))/n for _ in range(samples))
    return {'n':n,'mean_s3_minus_s1':statistics.mean(differences),
            'ci95':[means[int(.025*samples)],means[int(.975*samples)]],
            'bootstrap_samples':samples,'seed':seed}
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
rows=[]
for item in manifest['results']:
    doi=item['doi']; row={'doi':doi,'subject':str(records[doi].get('subject') or ''),
        'status':item['status'],'seconds':item.get('seconds')}
    if item['status']=='completed':
        pair=json.loads(Path(item['path']).read_text(encoding='utf-8'))
        assert (pair['doi']==doi and pair['revision']==DATA_REVISION
                and pair['selection_sha256']==SELECTION_SHA256
                and pair['test_content_accessed'] is False)
        abstract=str(records[doi].get('abstract') or '').strip()
        row.update({'abstract_words':len(abstract.split()),'source_tokens':pair['source_tokens'],
          'prompt_imbalance_tokens':pair['prompt_imbalance_tokens'],
          'S1_map_gpu_batches':pair['arms']['S1']['map_gpu_batches'],
          'S3_map_gpu_batches':pair['arms']['S3']['map_gpu_batches'],
          'S1_map_cap_hits':sum(call['output_tokens']>=config.max_output_tokens
                              for call in pair['arms']['S1']['calls'][:-1]),
          'S1_final_cap_hit':pair['arms']['S1']['calls'][-1]['output_tokens']>=config.final_output_tokens,
          'S3_map_cap_hits':sum(call['output_tokens']>=config.max_output_tokens
                              for call in pair['arms']['S3']['calls'][:-1]),
          'S3_final_cap_hit':pair['arms']['S3']['calls'][-1]['output_tokens']>=config.final_output_tokens,
          'S1_ROUGE_L':rouge_l_f1(abstract,pair['arms']['S1']['summary']) if abstract else None,
          'S3_ROUGE_L':rouge_l_f1(abstract,pair['arms']['S3']['summary']) if abstract else None})
        if row['S1_ROUGE_L'] is not None:
            row['delta_S3_minus_S1']=row['S3_ROUGE_L']-row['S1_ROUGE_L']
    else:
        row['reason']=item.get('reason')
    rows.append(row)
frame=pd.DataFrame(rows)
display(frame)
differences=[row['delta_S3_minus_S1'] for row in rows if 'delta_S3_minus_S1' in row]
completed=sum(row['status']=='completed' for row in rows)
subject_summary=[]
for subject in sorted(frame['subject'].unique()):
    group=frame[frame['subject']==subject]
    deltas=group['delta_S3_minus_S1'].dropna() if 'delta_S3_minus_S1' in group else []
    subject_summary.append({'subject':subject,'declared':len(group),
        'completed':int(group['status'].eq('completed').sum()),
        'scored':len(deltas),'mean_delta':float(deltas.mean()) if len(deltas) else None})
length_sensitivity={}
for label,predicate in [('abstract_under_30',lambda n:n<30),
                        ('abstract_at_least_30',lambda n:n>=30)]:
    subset=[row['delta_S3_minus_S1'] for row in rows
            if 'delta_S3_minus_S1' in row and predicate(row['abstract_words'])]
    length_sensitivity[label]=paired_ci(subset)
result={'status':'complete' if completed==70 else 'incomplete_diagnostic',
 'dataset':'gigant/tib','revision':DATA_REVISION,'split':'valid',
 'test_content_accessed':False,'model':backend.model_id,
 'selection_sha256':SELECTION_SHA256,'n_declared':70,'n_completed':completed,
 'n_failed':sum(row['status']=='failed' for row in rows),
 'n_not_run':sum(row['status']=='not_run' for row in rows),
 'metric':'fallback_lcs_rouge_l_f1_against_unverified_creator_abstract',
 'paired_rouge':paired_ci(differences),'subject_summary':subject_summary,
 'abstract_length_sensitivity':length_sensitivity,'rows':rows,
 'limits':{'factuality_measured':False,'reference_validity_verified':False,
           'sealed_test_result':False,'image_audio_evaluated':False}}
score_path=output_dir/'automatic_evaluation.json'
score_path.write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Completed:',completed,'/70; failed:',result['n_failed'],'not run:',result['n_not_run'])
print('Paired ROUGE:',result['paired_rouge'])
display(pd.DataFrame(subject_summary))
print('Abstract-length sensitivity:',length_sensitivity)
print('Final cap hits:',sum(bool(row.get('S1_final_cap_hit')) for row in rows),'S1;',
      sum(bool(row.get('S3_final_cap_hit')) for row in rows),'S3')
print('Map GPU batches:',sum(row.get('S1_map_gpu_batches',0) for row in rows),'S1;',
      sum(row.get('S3_map_gpu_batches',0) for row in rows),'S3')
if differences:
    scored=frame.dropna(subset=['delta_S3_minus_S1'])
    fig,ax=plt.subplots(figsize=(8,3.5))
    ax.hist(scored['delta_S3_minus_S1'],bins=15)
    ax.axvline(0,color='black',linewidth=1)
    ax.set(xlabel='S3 minus S1 ROUGE-L F1',ylabel='Talks',
           title='Validation only; noisy abstract proxy')
    fig.tight_layout();display(fig);plt.close(fig)
print('Saved:',score_path)


## Sau khi chạy

Tải thư mục `MyDrive/RQ2_TIB_VALID_QWEN/full70_chat_*/` gồm `manifest.json`, các receipt DOI, `automatic_evaluation.json` và `source_reference_70_valid.json`. Gửi ZIP đó để tôi kiểm tra kết quả và đối chiếu câu tóm tắt với transcript. Dữ liệu transcript ở Drive; không thêm vào Git.
